# E16 prediction — wav + transcript → phoneme JSON

Inference with the locked **E16** scorer (Transformer + SSL phone embed on C8 78-d LPP+LPR).

**Pipeline:** transcript → `g2p_en` (CMU) → XLSR-53 espeak CTC → CTC-Viterbi 78-d → `checkpoints/e16/transformer_ckpt.pt` → scores `[0, 2]`.

Prerequisites:
- Checkpoint: `checkpoints/e16/transformer_ckpt.pt` (written by Group E `--features c8_lpp_lpr_embed`)
- AM: `models/wav2vec2-xlsr-53-espeak-cv-ft/`
- `pip install g2p-en soundfile` (and `librosa` if wav ≠ 16 kHz)
- NLTK data for g2p_en: repo already has `models/_scratch/nltk/` (cmudict); helper sets `NLTK_DATA` automatically

```text
python scripts/run_experiment.py --config configs/e_learned_scoring.yaml --features c8_lpp_lpr_embed
```

Free-form G2P ≠ Speechocean762 expert phones — do not claim PCC 0.67 on arbitrary text.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if not (ROOT / "src" / "gop_empirical").is_dir():
    ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

CKPT = ROOT / "checkpoints" / "e16" / "transformer_ckpt.pt"
AM = ROOT / "models" / "wav2vec2-xlsr-53-espeak-cv-ft"
print("ROOT", ROOT)
print("checkpoint", CKPT.exists(), CKPT)
print("AM", AM.is_dir(), AM)

ROOT D:\FSB_MSE\FINAL\GOP-Empirical-Study
checkpoint True D:\FSB_MSE\FINAL\GOP-Empirical-Study\outputs\E\e16_phone_transformer.pt
AM True D:\FSB_MSE\FINAL\GOP-Empirical-Study\models\wav2vec2-xlsr-53-espeak-cv-ft


In [2]:
# Ensure checkpoint exists (train+save once if missing).
if not CKPT.is_file():
    import subprocess

    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "export_group_e_checkpoint.py"),
        "--config",
        str(ROOT / "configs" / "e_learned_scoring.yaml"),
        "--experiment",
        "E16",
        "--features",
        "c8_lpp_lpr_embed",
        "--device",
        "cpu",
    ]
    print("training E16 checkpoint…")
    subprocess.check_call(cmd, cwd=str(ROOT))
assert CKPT.is_file(), CKPT

In [3]:
from gop_empirical.inference.e16 import dumps_result, run_e16_utterance
from gop_empirical.scoring.checkpoint import load_checkpoint

DEVICE = "cuda"  # or "cpu"
ckpt_meta = load_checkpoint(CKPT, device="cpu")
print(
    ckpt_meta["experiment_id"],
    ckpt_meta["architecture"],
    ckpt_meta["feature_set"],
    "fit",
    ckpt_meta.get("fit"),
    "test_pcc",
    (ckpt_meta.get("metrics") or {}).get("test", {}).get("pcc"),
)

E16 transformer c8_lpp_lpr_embed fit {'best_epoch': 14, 'best_val_mse': 0.05078276980965255, 'epochs_ran': 22} test_pcc 0.6673445603974063


## Input

Set `WAV_PATH` and `TRANSCRIPT`, then run the next cell.

In [4]:
# Demo Speechocean762 utterance (change paths as needed).
WAV_PATH = ROOT / "data" / "so762_inputs" / "wavs" / "000010011.wav"
TRANSCRIPT = "WE CALL IT BEAR"

print("wav", WAV_PATH.exists(), WAV_PATH)
print("text", TRANSCRIPT)

wav True D:\FSB_MSE\FINAL\GOP-Empirical-Study\data\so762_inputs\wavs\000010011.wav
text WE CALL IT BEAR


In [5]:
result = run_e16_utterance(
    WAV_PATH,
    TRANSCRIPT,
    checkpoint_path=CKPT,
    package_root=ROOT,
    device=DEVICE if DEVICE == "cpu" or __import__("torch").cuda.is_available() else "cpu",
)
print(dumps_result(result))

c:\miniconda3\envs\gop\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 424/424 [00:00<00:00, 26453.04it/s]


{
  "model": "E16",
  "acoustic_model": "wav2vec2-xlsr-53-espeak-cv-ft",
  "transcript": "WE CALL IT BEAR",
  "wav_path": "D:\\FSB_MSE\\FINAL\\GOP-Empirical-Study\\data\\so762_inputs\\wavs\\000010011.wav",
  "checkpoint": "D:\\FSB_MSE\\FINAL\\GOP-Empirical-Study\\outputs\\E\\e16_phone_transformer.pt",
  "g2p_phones": [
    "W",
    "IY",
    "K",
    "AO",
    "L",
    "IH",
    "T",
    "B",
    "EH",
    "R"
  ],
  "phones": [
    {
      "phone_id": 0,
      "phone": "W",
      "ssl_index": 35,
      "n_frames": 1,
      "score": 2.0
    },
    {
      "phone_id": 1,
      "phone": "IY",
      "ssl_index": 17,
      "n_frames": 1,
      "score": 1.9745824337005615
    },
    {
      "phone_id": 2,
      "phone": "K",
      "ssl_index": 19,
      "n_frames": 1,
      "score": 2.0
    },
    {
      "phone_id": 3,
      "phone": "AO",
      "ssl_index": 3,
      "n_frames": 1,
      "score": 1.9826462268829346
    },
    {
      "phone_id": 4,
      "phone": "L",
      "ssl_index": 20

In [6]:
# Optional: write JSON next to the wav.
out_json = Path(WAV_PATH).with_suffix(".e16.json")
out_json.write_text(dumps_result(result), encoding="utf-8")
print("wrote", out_json)
print("n_phones", len(result["phones"]))
print("mean_score", sum(p["score"] for p in result["phones"]) / max(len(result["phones"]), 1))

wrote D:\FSB_MSE\FINAL\GOP-Empirical-Study\data\so762_inputs\wavs\000010011.e16.json
n_phones 10
mean_score 1.9704160571098328


In [8]:
# So sánh với điểm chuyên gia Speechocean762 (scores.json) khi wav thuộc SO762.
# Ghép theo thứ tự phone (strip stress). G2P có thể lệch annotation — kiểm tra cột match.

import re

import pandas as pd

utt_id = Path(WAV_PATH).stem
scores_path = ROOT / "data" / "scores.json"
payload = json.loads(scores_path.read_text(encoding="utf-8"))
if utt_id not in payload:
    raise KeyError(f"{utt_id} not in {scores_path}; chỉ so được khi wav là utterance SO762")

stress_re = re.compile(r"[0-2]$")
human_rows = []
for w_i, word in enumerate(payload[utt_id]["words"]):
    phones = word["phones"]
    accs = word["phones-accuracy"]
    for p_i, (ph, acc) in enumerate(zip(phones, accs)):
        human_rows.append(
            {
                "word_id": w_i,
                "phone_id": p_i,
                "phone_human": stress_re.sub("", str(ph).upper()),
                "human_score": float(acc),
                "word": word.get("text", ""),
            }
        )

pred_rows = [
    {"phone_id_seq": i, "phone_pred": p["phone"], "pred_e16": p["score"]}
    for i, p in enumerate(result["phones"])
]
n = min(len(human_rows), len(pred_rows))
cmp = pd.DataFrame(
    {
        "phone_id": list(range(n)),
        "word": [human_rows[i]["word"] for i in range(n)],
        "phone_human": [human_rows[i]["phone_human"] for i in range(n)],
        "phone_pred": [pred_rows[i]["phone_pred"] for i in range(n)],
        "human_score": [human_rows[i]["human_score"] for i in range(n)],
        "pred_e16": [pred_rows[i]["pred_e16"] for i in range(n)],
    }
)
cmp["match"] = cmp["phone_human"] == cmp["phone_pred"]
cmp["abs_err"] = (cmp["pred_e16"] - cmp["human_score"]).abs()

print(f"utt_id={utt_id}  n_human={len(human_rows)}  n_pred={len(pred_rows)}  aligned={n}")
print(cmp.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print()
print(
    "MAE",
    float(cmp["abs_err"].mean()),
    "| mean human",
    float(cmp["human_score"].mean()),
    "| mean pred",
    float(cmp["pred_e16"].mean()),
    "| phone match",
    f"{int(cmp['match'].sum())}/{n}",
)


utt_id=000010011  n_human=10  n_pred=10  aligned=10
 phone_id word phone_human phone_pred  human_score  pred_e16  match  abs_err
        0   WE           W          W        2.000     2.000   True    0.000
        1   WE          IY         IY        2.000     1.975   True    0.025
        2 CALL           K          K        2.000     2.000   True    0.000
        3 CALL          AO         AO        1.800     1.983   True    0.183
        4 CALL           L          L        1.800     1.923   True    0.123
        5   IT          IH         IH        2.000     2.000   True    0.000
        6   IT           T          T        2.000     1.990   True    0.010
        7 BEAR           B          B        2.000     2.000   True    0.000
        8 BEAR          EH         EH        1.000     1.902   True    0.902
        9 BEAR           R          R        1.000     1.932   True    0.932

MAE 0.21759584188461303 | mean human 1.7600000000000002 | mean pred 1.9704160571098328 | phone match